# Orange County Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import statements
import pandas as pd
import src.data_utils as data_utils
import src.rate_utils as rate_utils
import src.table_utils as table_utils
import src.visualization_utils as visualization_utils

# Display all columns
pd.set_option('display.max_columns', None)

# Import Data

In [ ]:
# Load in policing data
original_policing = pd.read_csv("../data/processed/cleaned_orange_ripa_2022_2024.csv", low_memory=False)

# Select only relevant columns to use
keep_cols = [
    "doj_record_id",
    "person_number",
    "agency_ori",
    "agency_name",
    "time_of_stop",
    "date_of_stop",
    "year",
    "stop_duration",
    "closest_city",
    "race_ethnicity",
    "gender",
    "age",
    "reason_for_contact",
    "traffic_violation_cjis_offense_code",
    "suspicion_cjis_offense_code",
    "suspicion",
    "action_any_search",
    "search_basis_plain_view",
    "search_basis_plain_smell",
    "search_basis_consent",
    "search_basis_safety",
    "search_basis_suspect_weapon",
    "search_basis_evidence_of_crime",
    "search_basis_school_policy",
    "search_basis_emergency",
    "search_basis_canine",
    "search_basis_warrant",
    "search_basis_probation",
    "search_basis_incident_to_arrest",
    "search_basis_vehicle_inventory",
    "contraband_any",
    "result_of_stop_arrest",
    "search_type",
    "multi_person_stop"
]
policing = original_policing[[c for c in keep_cols if c in original_policing.columns]].copy()

In [ ]:
# Load in prosecution data
prosecution = pd.read_csv("https://raw.githubusercontent.com/laurenbchu/honors-thesis/main/data/processed/cleaned_orange_aclu_2021_2023.csv", low_memory=False)
prosecution['filed_date'] = pd.to_datetime(prosecution["filed_date"])
prosecution['Year'] = prosecution['filed_date'].dt.year

In [ ]:
# Load in census data
census = data_utils.load_census("06059")

# Roll-up census data to be more coarse to match policing/prosecution race categories
census_coarse = data_utils.census_rollup(census)

# Policing Analysis: Stop Rates, Search Rates, and Hit Rates

In [ ]:
# Filter for only discretionary stops
policing = policing[policing["reason_for_contact"].isin([
    "Moving violation",
    "Equipment violation",
    "Non-moving violation",
    "Suspect criminal activity"
])].copy()

# Add standardized race column to match policing and prosecution labels
policing = data_utils.add_standardized_race(policing, "race_ethnicity")

# Consider only purely discretionary searches in search and hit rates

# Clean contraband_any (handles NaN, 0/1, True/False)
policing["contraband_any"] = policing["contraband_any"].fillna(0).astype(int)

# Create discretionary search indicator
policing["disc_search"] = policing["action_any_search"] & policing["search_type"].eq("Discretionary only")

# Create hits among discretionary searches indicator
# Hits only count as hits if they were found in a search, and the search was discretionary
policing["disc_hit"] = (
    policing["disc_search"] & # Has to be from a discretionary search
    policing["action_any_search"].fillna(False) & # Has to have been searched
    (policing["contraband_any"] == 1)
)

# Create a copy with all people involved in stops for sensitivity analysis
policing_multi = policing.copy()

# Filter for person 1 for each stop (usually, the main person stopped, but not always)
policing = policing[policing["person_number"] == 1].copy()

policing_analysis = rate_utils.policing_rates(policing, census_coarse)

In [ ]:
# Creating tables
race_order = ["Black/African American", "Hispanic/Latino", "White", "Asian", "Other"]

df = policing_analysis.copy()

pop = (df[["Perceived Race", "Population"]]
       .drop_duplicates("Perceived Race")
       .set_index("Perceived Race")
       .reindex(race_order))

stops_1k = table_utils.make_pivot_with_ci(df, "Stops per 1,000", "Stops per 1,000 SE")
searches_1k = table_utils.make_pivot_with_ci(df, "Searches per 1,000", "Searches per 1,000 SE")

In [ ]:
table1 = pd.concat(
    [pop, stops_1k, searches_1k],
    axis=1,
    keys=["Population", "Stops per 1,000", "Searches per 1,000"]
)


table1.columns = pd.MultiIndex.from_tuples(
    [(g, c) for g, c in table1.columns],
    names=["", "Year"]
)

table1
# export_stops_searches_to_latex(policing_analysis)

In [ ]:
tab2 = df.copy()

# Convert proportions to percentages
tab2["Search Rate"] *= 100
tab2["Hit Rate"] *= 100
tab2["Search Rate SE"] *= 100
tab2["Hit Rate SE"] *= 100

search_rate = table_utils.make_pivot_with_ci(tab2, "Search Rate", "Search Rate SE")
hit_rate = table_utils.make_pivot_with_ci(tab2, "Hit Rate", "Hit Rate SE")

table2 = pd.concat([search_rate, hit_rate], axis=1, keys=["Search Rate (%)", "Hit Rate (%)"])

table2.columns = pd.MultiIndex.from_tuples(
    [(g, c) for g, c in table2.columns],
    names=["", "Year"]
)

table2 = table2.reindex(race_order)
table2
# export_searches_hits_to_latex(policing_analysis)

In [ ]:
policing_figs = visualization_utils.visualize_policing(policing_analysis)
# visualization_utils.export_figure_to_pdf(policing_figs, "policing")

# Policing Analysis: Stratified by Reason for Contact

In [ ]:
# Calculate rates by reason for contact
policing_by_reason = rate_utils.policing_rates_by_reason_for_contact(policing, census_coarse)

# Create comprehensive table with pooled 2022-2024 data
reason_table = table_utils.create_reason_for_contact_table(policing_by_reason)
display(reason_table)
# table_utils.export_reason_for_contact_table_to_latex(reason_table)

In [ ]:
# Create search rate visualization
fig_search = visualization_utils.visualize_search_and_hit_rates_by_reason(policing_by_reason)
# visualization_utils.export_figure_to_pdf(fig_search, "policing_by_reason")

# Policing Analysis: Black and White Hit Rates by Agency

In [ ]:
# Calculate the White and Black hit rates for each agency and summarize in a table
agency_hit_summary = rate_utils.summarize_agency_black_white_hit_rates(policing)
publication_table = table_utils.agency_black_white_hit_rate_table(agency_hit_summary)
publication_table
# table_utils.export_agency_hit_rates_to_latex(agency_hit_summary)

In [ ]:
fig_agency_hit = visualization_utils.plot_agency_black_white_hit_rates(agency_hit_summary)
# visualization_utils.export_figure_to_pdf(fig_agency_hit, "agency_hits")

# Policing Sensitivity Analysis: Mixed Search Bases and Multiperson Stops

In [ ]:
# Checks sensitivity of search and hit rates to including searches with mixed legal bases (e.g. "Mixed" or "No search basis") as discretionary searches

policing["disc_search_mixed"] = (
    policing["action_any_search"]
    & policing["search_type"].isin(["Discretionary only", "Mixed", "No search basis"])
)

policing["disc_hit_mixed"] = (
    policing["disc_search_mixed"]
    & policing["action_any_search"].fillna(False)
    & (policing["contraband_any"] == 1)
)

policing_analysis_mixed = rate_utils.policing_rates_sensitivity(
    policing,
    census_coarse,
    "mixed"
)

# Checks sensitivity of search and hit rates to including all people involved in stops (not just person 1) for multiperson stops

policing_multi.rename(
    columns={"disc_search": "disc_search_multiperson"},
    inplace=True
)

policing_multi["disc_hit_multiperson"] = (
    policing_multi["disc_search_multiperson"]
    & policing_multi["action_any_search"].fillna(False)
    & (policing_multi["contraband_any"] == 1)
)

policing_analysis_multiperson = rate_utils.policing_rates_sensitivity(
    policing_multi,
    census_coarse,
    "multiperson"
)

In [ ]:
# Creates one table for 2024

def make_2024_block(df, label):
    """Return a 2024 table with formatted Search/Hit rates for one method."""
    d = df[df["Year"] == 2024].copy()
    d = d.set_index("Perceived Race")

    def special_fmt_est_ci(rate, se, digits=2):
        pct = rate * 100
        ci = 1.96 * se * 100
        return f"{pct:.{digits}f} (±{ci:.{digits}f})"

    out = pd.DataFrame(index=d.index)
    out[("Search Rate (%)", label)] = [
        special_fmt_est_ci(e, s, 2)
        for e, s in zip(d["Search Rate"], d["Search Rate SE"])
    ]
    out[("Hit Rate (%)", label)] = [
        special_fmt_est_ci(e, s, 2)
        for e, s in zip(d["Hit Rate"], d["Hit Rate SE"])
    ]
    return out

# Build 2024 display blocks
baseline_2024 = make_2024_block(policing_analysis.copy(), "Baseline")
mixed_2024 = make_2024_block(policing_analysis_mixed.copy(), "Mixed")
multiperson_2024 = make_2024_block(policing_analysis_multiperson.copy(), "Multiperson")

# Join into one combined display table
table3 = baseline_2024.join(mixed_2024, how="outer").join(multiperson_2024, how="outer")

# Enforce row order
race_order = ["Black/African American", "Hispanic/Latino", "White", "Asian", "Other"]
table3 = table3.reindex(race_order)

# Enforce column order
table3 = table3[
    [
        ("Search Rate (%)", "Baseline"),
        ("Search Rate (%)", "Mixed"),
        ("Search Rate (%)", "Multiperson"),
        ("Hit Rate (%)", "Baseline"),
        ("Hit Rate (%)", "Mixed"),
        ("Hit Rate (%)", "Multiperson"),
    ]
]

table3.columns = pd.MultiIndex.from_tuples(table3.columns, names=["", ""])
table3.index.name = "Perceived Race"

table3
# export_combined_sensitivity_to_latex(policing_analysis, policing_analysis_mixed, policing_analysis_multiperson)

In [ ]:
sensitivity_fig = visualization_utils.create_combined_sensitivity_visualization(
    policing_analysis,
    policing_analysis_mixed,
    policing_analysis_multiperson
)

# visualization_utils.export_figure_to_pdf(sensitivity_fig, "sensitivity")

# Prosecution Analysis: Enhancement Rates by Charge and Statute Level

In [ ]:
# Harmonizes race labels
prosecution = data_utils.add_standardized_race(prosecution, 'canonical_race')

# Identify charges that are enhancements/priors/sentencing considerations rather than actual substantive criminal charges
prosecution['is_non_substantive'] = (
    prosecution['is_enhancement_charge'].astype(bool) |
    prosecution['is_special_circumstance'].astype(bool) |
    prosecution['is_sentencing_charge'].astype(bool)
)

# Apply categorization ONLY to substantive charges
prosecution.loc[~prosecution['is_non_substantive'], 'charge_category'] = \
    prosecution.loc[~prosecution['is_non_substantive'], 'charge_description'].apply(data_utils.categorize_charge)

# For non-substantive charges, label them explicitly as ehancement/prior/sentencing
prosecution.loc[prosecution['is_non_substantive'], 'charge_category'] = 'Enhancement/Prior/Sentencing'

# Each case is a single defendant, so case-level = defendant-case-level
# For each case, if any charge is an enhancement charge, then the case is flagged as having an enhancement charge
enh_flag = (
    prosecution.groupby('source_case_id')['is_enhancement_charge']
    .max()
    .rename('any_enhancement_in_case')
    .reset_index()
)

# Drop the enhancement rows, keeping only the base charges
substantive = prosecution[~prosecution['is_non_substantive']].copy()

# Add the enhancement flag to the dataframe
substantive = substantive.merge(enh_flag, on='source_case_id', how='left')
substantive['any_enhancement_in_case'] = substantive['any_enhancement_in_case'].fillna(0).astype(int)

# Calculate enhancement rates by charge category
# For each case, a primary charge category and statute level is assigned
# Then computes cases with any enhancement charge over total cases for each charge category
enhancement_by_primary = rate_utils.enhancement_rates_by_primary_severity(substantive)

# Calculate and display average enhancement rates by primary charge category
enhancement_by_primary.groupby("primary_charge_category").apply(lambda g: g["Enhanced"].sum() / g["N"].sum()).sort_values(ascending=False)

# Excludes the "Other" category and categories with less than a 5% average enhancement rate across all races and statute levels
filtered_enhancement = enhancement_by_primary[
    enhancement_by_primary["primary_charge_category"].isin(
        enhancement_by_primary.groupby("primary_charge_category")
        .apply(lambda g: g["Enhanced"].sum() / g["N"].sum())
        .loc[lambda s: (s >= 0.05) & (s.index != "Other")]
        .index
    )
].copy()

In [ ]:
# Table 1: By race and statute level
enhancement_combined = table_utils.enhancement_rate_combined_table(enhancement_by_primary)
display(enhancement_combined)

# Table 2: By race, statute level, and primary category (for all categories)
# Excludes the "Other" category and categories with less than a 5% overall enhancement rate
enhancement_category = table_utils.enhancement_rate_race_statute_category_table(filtered_enhancement)
display(enhancement_category)

# export_enhancement_tables_to_latex(enhancement_combined, enhancement_category)

In [ ]:
# Plot: By race and statute level

without_other = enhancement_by_primary[enhancement_by_primary["race_std"] != "Other"].copy()
enhancement_rate_race_statute = visualization_utils.plot_enhancement_rate_by_race_statute(without_other)
# visualization_utils.export_figure_to_pdf(enhancement_rate_race_statute, "enhancement_rate_by_race_statute")

In [ ]:
figs = visualization_utils.plot_enhancement_rate_by_race_statute_category(enhancement_by_primary)
# visualization_utils.export_figure_to_pdf(figs["assault_violence_weapons"], "enhancement_assault_violence_weapons")
# visualization_utils.export_figure_to_pdf(figs["dui"], "enhancement_dui")

# Prosecution Analysis: Wobblers

In [ ]:
filtered_wobbler_categories = rate_utils.get_filtered_wobbler_categories(substantive)
wobbler_table, wobbler_category_table = table_utils.wobbler_felony_rate_tables(substantive)

display(wobbler_table)

# For the category table, we only display categories with at least 5% overall wobbler felony filing rate and total wobbler cases of at least 500
# Also exclude "Other" category
display(wobbler_category_table)

# export_wobbler_tables_to_latex(wobbler_table, wobbler_category_table)

In [ ]:
fig = visualization_utils.plot_wobbler_combined(substantive, top_categories=filtered_wobbler_categories, sort_by="rate_overall")
# visualization_utils.export_figure_to_pdf(fig, "wobblers")